# Milestone 1: Exploratory Data Analysis & Data Pipeline

**Module:** MSc Deep Learning Applications (CMP-L016)  
**Project #28:** Weather Prediction with Hybrid Deep Learning Models  
**Dataset:** Jena Climate 2009–2022 (Max Planck Institute for Biogeochemistry)

---

## Table of Contents
1. [Setup & Imports](#1)
2. [Dataset Loading](#2)
3. [Data Inspection](#3)
4. [Time Series Visualisation](#4)
5. [Seasonal Decomposition](#5)
6. [Correlation Analysis](#6)
7. [Distribution Analysis](#7)
8. [Stationarity Testing](#8)
9. [Autocorrelation Analysis](#9)
10. [Data Preprocessing Pipeline](#10)
11. [Summary](#11)

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Ensure we're in the project root (handles running from notebooks/ dir)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f"Working directory: {os.getcwd()}")

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10
sns.set_palette("husl")

FIGURE_DIR = os.path.join("outputs", "figures")
os.makedirs(FIGURE_DIR, exist_ok=True)
print("Setup complete!")

## 2. Dataset Loading

The **Jena Climate dataset** contains weather observations recorded every **10 minutes** 
at the Max Planck Institute for Biogeochemistry in Jena, Germany from 2009–2022.

**14 meteorological features:**
- Temperature (°C), Atmospheric pressure (mbar)
- Humidity (%), Saturation vapor pressure, Vapor pressure deficit
- Wind speed (m/s), Wind direction (°)
- And more — covering a comprehensive set of atmospheric measurements

**Citation:** Max Planck Institute for Biogeochemistry, Jena, Germany

In [ ]:
CSV_PATH = os.path.join("data", "raw", "jena_climate_2009_2016.csv")
df = pd.read_csv(CSV_PATH)

# Parse datetime
df['Date Time'] = pd.to_datetime(df['Date Time'], format='%d.%m.%Y %H:%M:%S')
df = df.set_index('Date Time').sort_index()

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df.index[0]} to {df.index[-1]}")
print(f"Duration: {(df.index[-1] - df.index[0]).days} days ({(df.index[-1] - df.index[0]).days / 365:.1f} years)")
print(f"Sampling rate: every 10 minutes")
print(f"\nFeatures:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

## 3. Data Inspection

In [ ]:
# Summary statistics
df.describe().round(2)

In [ ]:
# Missing values
missing = df.isnull().sum()
if missing.sum() > 0:
    print(f"Total missing values: {missing.sum()}")
    print(missing[missing > 0])
else:
    print("✓ No missing values found!")

# Check for anomalies
print(f"\nPotential anomalies:")
print(f"  Temperature range: {df['T (degC)'].min():.1f}°C to {df['T (degC)'].max():.1f}°C")
print(f"  Pressure range: {df['p (mbar)'].min():.1f} to {df['p (mbar)'].max():.1f} mbar")
print(f"  Wind speed max: {df['wv (m/s)'].max():.1f} m/s")
print(f"  Max wind speed max: {df['max. wv (m/s)'].max():.1f} m/s")

# Check for erroneous values (e.g., wind speed = -9999)
for col in df.columns:
    n_neg = (df[col] < -900).sum()
    if n_neg > 0:
        print(f"  ⚠ {col}: {n_neg} suspicious values (< -900)")

In [ ]:
# Fix erroneous wind speed values (some datasets have -9999 as missing indicator)
for col in ['wv (m/s)', 'max. wv (m/s)']:
    if (df[col] < 0).any():
        n_bad = (df[col] < 0).sum()
        df.loc[df[col] < 0, col] = 0
        print(f"Fixed {n_bad} negative values in '{col}' → set to 0")

# Resample to hourly (reduces 420K rows to ~70K for faster processing)
df_hourly = df.resample('1h').mean()
print(f"\nResampled to hourly: {df_hourly.shape[0]:,} rows")
print(f"This reduces computation by {len(df)/len(df_hourly):.0f}× while preserving all patterns")

## 4. Time Series Visualisation

### 4.1 Full Temperature Timeline

The 7-year temperature record shows clear annual cycles with amplitude ~35°C.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_hourly.index, df_hourly['T (degC)'], linewidth=0.2, alpha=0.6, color='steelblue')
rolling = df_hourly['T (degC)'].rolling(window=24*30).mean()
ax.plot(df_hourly.index, rolling, linewidth=2, color='red', label='30-day Moving Average')
ax.set_xlabel('Date'); ax.set_ylabel('Temperature (°C)')
ax.set_title('Jena Temperature (2009–2022) — Full Timeline', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'eda_temperature_timeline.png'), dpi=150, bbox_inches='tight')
plt.show()

### 4.2 All Features Overview

In [ ]:
fig, axes = plt.subplots(7, 2, figsize=(16, 24), sharex=True)
axes = axes.flatten()
for i, col in enumerate(df_hourly.columns):
    ax = axes[i]
    ax.plot(df_hourly.index, df_hourly[col], linewidth=0.2, alpha=0.7, color='steelblue')
    ax.set_ylabel(col, fontsize=8)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.grid(True, alpha=0.3)
plt.suptitle('Jena Climate — All 14 Features (Hourly)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'eda_all_features.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Seasonal Decomposition

### 5.1 Monthly Temperature Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly boxplots
monthly_data = [df_hourly.loc[df_hourly.index.month == m, 'T (degC)'].values for m in range(1, 13)]
bp = axes[0].boxplot(monthly_data, labels=[
    'Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'
], patch_artist=True)
colors_bp = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, 12))
for patch, c in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[0].set_xlabel('Month'); axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('Monthly Temperature Distribution', fontweight='bold')

# Hourly averages (diurnal cycle)
hourly_avg = df_hourly.groupby(df_hourly.index.hour)['T (degC)'].agg(['mean','std'])
axes[1].plot(hourly_avg.index, hourly_avg['mean'], 'o-', color='steelblue', lw=2)
axes[1].fill_between(hourly_avg.index, hourly_avg['mean']-hourly_avg['std'],
                      hourly_avg['mean']+hourly_avg['std'], alpha=0.2, color='steelblue')
axes[1].set_xlabel('Hour of Day'); axes[1].set_ylabel('Temperature (°C)')
axes[1].set_title('Diurnal Temperature Cycle (Mean ± Std)', fontweight='bold')
axes[1].set_xticks(range(0, 24, 3))

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'eda_seasonal_patterns.png'), dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Year-over-Year Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for year in range(2009, 2017):
    mask = df_hourly.index.year == year
    yearly = df_hourly.loc[mask, 'T (degC)'].resample('D').mean()
    day_of_year = yearly.index.dayofyear
    ax.plot(day_of_year, yearly.values, alpha=0.6, lw=1, label=str(year))
ax.set_xlabel('Day of Year'); ax.set_ylabel('Temperature (°C)')
ax.set_title('Year-over-Year Temperature Comparison', fontweight='bold')
ax.legend(ncol=4, fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'eda_yearly_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation Analysis

The heatmap reveals strong correlations between temperature-related variables 
(T, Tdew, Tpot) and humidity-related variables (rh, VPmax, VPact, VPdef).

In [ ]:
corr = df_hourly.corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, square=True, linewidths=0.5,
            annot=True, fmt='.2f', cbar_kws={"shrink": 0.8}, ax=ax, annot_kws={"size": 7})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'eda_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with temperature
print("Top correlations with T (degC):")
temp_corr = corr['T (degC)'].drop('T (degC)').abs().sort_values(ascending=False)
for feat, val in temp_corr.items():
    print(f"  {feat:<20s}: {corr['T (degC)'][feat]:>7.4f}")

## 7. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(df_hourly.columns):
    ax = axes[i]
    data = df_hourly[col].dropna()
    ax.hist(data, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='black', lw=0.3)
    ax.axvline(data.mean(), color='red', lw=1.5, ls='--', label=f'Mean: {data.mean():.1f}')
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.legend(fontsize=6)
# Hide extra subplot
axes[-1].set_visible(False)
plt.suptitle('Distribution of All Weather Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'eda_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Stationarity Testing (ADF)

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Test on daily means (faster than hourly)
df_daily = df_hourly.resample('D').mean()

print("Augmented Dickey-Fuller Test Results (Daily Data):")
print(f"{'Feature':<22s} {'ADF Stat':>10s} {'p-value':>12s} {'Result':>12s}")
print("-" * 60)
for col in df_daily.columns:
    try:
        result = adfuller(df_daily[col].dropna(), autolag='AIC')
        is_stat = result[1] < 0.05
        print(f"  {col:<20s} {result[0]:>10.4f} {result[1]:>12.6f} {'Stationary' if is_stat else 'Non-stat':>12s}")
    except:
        print(f"  {col:<20s} {'Error':>10s}")

## 9. Autocorrelation Analysis

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
plot_acf(df_daily['T (degC)'].dropna(), lags=90, ax=axes[0],
         title='Temperature — Autocorrelation (90 lags)')
plot_pacf(df_daily['T (degC)'].dropna(), lags=90, ax=axes[1],
          title='Temperature — Partial Autocorrelation (90 lags)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'eda_autocorrelation.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Data Preprocessing Pipeline

The pipeline prepares data for model training:
1. **Resample** to hourly (reduce 420K → 70K rows)
2. **Clean** erroneous wind speed values
3. **Chronological split** — Train 70% / Val 15% / Test 15%
4. **Normalise** — StandardScaler fitted on training data only (no leakage)
5. **Windowing** — 168 hours (7 days) input → next-hour temperature prediction

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Preprocessing pipeline
df_clean = df_hourly.copy()
df_clean = df_clean.interpolate(method='linear').ffill().bfill()

feature_names = list(df_clean.columns)
target_col = 'T (degC)'
target_idx = feature_names.index(target_col)
print(f"Features: {len(feature_names)}")
print(f"Target: {target_col} (index {target_idx})")

# Chronological split
n = len(df_clean)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_df = df_clean.iloc[:train_end]
val_df = df_clean.iloc[train_end:val_end]
test_df = df_clean.iloc[val_end:]

print(f"\nTrain: {len(train_df):,} ({train_df.index[0]} → {train_df.index[-1]})")
print(f"Val:   {len(val_df):,} ({val_df.index[0]} → {val_df.index[-1]})")
print(f"Test:  {len(test_df):,} ({test_df.index[0]} → {test_df.index[-1]})")

# Normalise
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

print(f"\nScaler fitted on training data only (mean={scaler.mean_[target_idx]:.2f}°C, std={scaler.scale_[target_idx]:.2f}°C)")
print(f"✓ Pipeline complete!")

## 11. Summary

### Key EDA Findings

| Finding | Detail |
|---------|--------|
| **Dataset size** | 420,551 observations (10-min intervals), 2009–2022 |
| **Resampled** | to 70,091 hourly observations for training efficiency |
| **Features** | 14 meteorological variables |
| **Target** | Temperature (°C) — strong seasonal and diurnal patterns |
| **Correlations** | High between T, Tdew, Tpot; moderate for humidity/pressure |
| **Stationarity** | Most features stationary (ADF test p < 0.05) |
| **Anomalies** | Fixed erroneous wind speed values (< 0 → 0) |

### Ready for Milestone 2
- Data pipeline is working ✓
- Features analysed and understood ✓
- Preprocessing validated (no data leakage) ✓